# Sentiment Analysis with DistilBERT — Trainer API

This is the Trainer-based version of `Lab1_DistilBERT.ipynb`. Training is delegated to Hugging Face `Trainer` + `TrainingArguments`; only class-weighted loss, the classification report, and the confusion matrix plot remain as custom helpers in `transformer_utils_v2.py`.

Datasets:
- 1K Amazon Reviews
- 25K Amazon Reviews
- Video Game Reviews (5-class, class-weighted)

## Setup & Imports

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import wandb

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from utils import device_check
from transformer_utils_v2 import (
    build_tf_datasets,
    compute_metrics_tf,
    WeightedTrainer,
    evaluate_tf,
    plot_confusion_matrix_tf,
)

device = device_check()

In [ ]:
LOG_WANDB = True
SEED      = 1

# Assumes the notebook is launched from Lab1/src.
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent
WANDB_DIR    = PROJECT_DIR
SPLITS_DIR   = PROJECT_DIR / 'data' / 'splits'
MODELS_DIR   = PROJECT_DIR / 'models'

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# wandb project/entity are read from env vars by Trainer and wandb.init.
os.environ['WANDB_PROJECT'] = 'Lab1'
os.environ['WANDB_ENTITY']  = 'd7047e-group12'
os.environ['WANDB_DIR']     = str(WANDB_DIR)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

---
## Initial Test — 1K Amazon Reviews

### Load Pre-split Data

In [ ]:
train_df = pd.read_csv(SPLITS_DIR / '1k_train.csv')
val_df   = pd.read_csv(SPLITS_DIR / '1k_val.csv')
test_df  = pd.read_csv(SPLITS_DIR / '1k_test.csv')

text_col  = 'Sentence'
label_col = 'Class'

NUM_LABELS  = 2
NUM_CLASSES = 2

### Tokenizer & Datasets

In [ ]:
datasets = build_tf_datasets(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    tokenizer=tokenizer,
    text_col=text_col,
    label_col=label_col,
    max_length=MAX_LENGTH,
)

### Model & Trainer

The `Trainer` handles the epoch loop, AMP, warmup scheduling, gradient clipping, best-checkpoint restoration, and wandb logging. `compute_metrics_tf` adds accuracy and macro/weighted F1 to every eval step.

In [ ]:
BATCH_SIZE    = 32
NUM_WORKERS   = min(4, os.cpu_count() or 0)
NUM_EPOCHS    = 5
LEARNING_RATE = 2e-5
WARMUP_RATIO  = 0.1
RUN_NAME      = 'DistilBERT 1K'
OUT_DIR       = MODELS_DIR / 'distilbert_1k'

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

wandb.init(
    name=RUN_NAME,
    tags=['Task 1.2', 'DistilBERT', 'Transformer', '1K', 'Trainer'],
    config={
        'dataset':      'Amazon 1K',
        'model_name':   MODEL_NAME,
        'optimizer':    'AdamW',
        'lr':           LEARNING_RATE,
        'epochs':       NUM_EPOCHS,
        'batch_size':   BATCH_SIZE,
        'max_length':   MAX_LENGTH,
        'warmup_ratio': WARMUP_RATIO,
    },
    reinit=True,
    mode='online' if LOG_WANDB else 'disabled',
)

training_args = TrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=1.0,
    fp16=torch.cuda.is_available(),
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_strategy='epoch',
    dataloader_num_workers=NUM_WORKERS,
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to=['wandb'] if LOG_WANDB else 'none',
    run_name=RUN_NAME,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=datasets['train'],
    eval_dataset=datasets['val'],
    compute_metrics=compute_metrics_tf,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(str(OUT_DIR))

wandb.finish()

### Test-Set Evaluation & Confusion Matrix

In [ ]:
_ = evaluate_tf(
    trainer,
    datasets['test'],
    label='DistilBERT-1K',
    class_names=['Negative', 'Positive'],
)

In [ ]:
plot_confusion_matrix_tf(
    trainer,
    datasets['test'],
    NUM_CLASSES,
    ['Negative', 'Positive'],
    'DistilBERT Confusion Matrix — 1K Samples',
    normalize=True,
)

---
## Scaling Up — 25K Amazon Reviews

### Load Pre-split Data

In [ ]:
train_df = pd.read_csv(SPLITS_DIR / '25k_train.csv')
val_df   = pd.read_csv(SPLITS_DIR / '25k_val.csv')
test_df  = pd.read_csv(SPLITS_DIR / '25k_test.csv')

text_col  = 'Sentence'
label_col = 'Class'

NUM_LABELS  = 2
NUM_CLASSES = 2

### Tokenizer & Datasets

In [ ]:
datasets = build_tf_datasets(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    tokenizer=tokenizer,
    text_col=text_col,
    label_col=label_col,
    max_length=MAX_LENGTH,
)

### Model & Trainer

In [ ]:
BATCH_SIZE    = 32
NUM_WORKERS   = min(4, os.cpu_count() or 0)
NUM_EPOCHS    = 5
LEARNING_RATE = 2e-5
WARMUP_RATIO  = 0.1
RUN_NAME      = 'DistilBERT 25K'
OUT_DIR       = MODELS_DIR / 'distilbert_25k'

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

wandb.init(
    name=RUN_NAME,
    tags=['Task 1.2', 'DistilBERT', 'Transformer', '25K', 'Trainer'],
    config={
        'dataset':      'Amazon 25K',
        'model_name':   MODEL_NAME,
        'optimizer':    'AdamW',
        'lr':           LEARNING_RATE,
        'epochs':       NUM_EPOCHS,
        'batch_size':   BATCH_SIZE,
        'max_length':   MAX_LENGTH,
        'warmup_ratio': WARMUP_RATIO,
    },
    reinit=True,
    mode='online' if LOG_WANDB else 'disabled',
)

training_args = TrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=1.0,
    fp16=torch.cuda.is_available(),
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_strategy='epoch',
    dataloader_num_workers=NUM_WORKERS,
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to=['wandb'] if LOG_WANDB else 'none',
    run_name=RUN_NAME,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=datasets['train'],
    eval_dataset=datasets['val'],
    compute_metrics=compute_metrics_tf,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(str(OUT_DIR))

wandb.finish()

### Test-Set Evaluation & Confusion Matrix

In [ ]:
_ = evaluate_tf(
    trainer,
    datasets['test'],
    label='DistilBERT-25K',
    class_names=['Negative', 'Positive'],
)

In [ ]:
plot_confusion_matrix_tf(
    trainer,
    datasets['test'],
    NUM_CLASSES,
    ['Negative', 'Positive'],
    'DistilBERT Confusion Matrix — 25K Samples',
    normalize=True,
)

---
## Video Game Reviews — 5-Class Rating

### Load Pre-split Data

In [ ]:
train_df = pd.read_csv(SPLITS_DIR / 'vg_train.csv')
val_df   = pd.read_csv(SPLITS_DIR / 'vg_val.csv')
test_df  = pd.read_csv(SPLITS_DIR / 'vg_test.csv')

text_col  = 'Sentence'
label_col = 'Class'

# Convert labels from 1-5 → 0-4
train_df[label_col] = train_df[label_col].astype(int) - 1
val_df[label_col]   = val_df[label_col].astype(int) - 1
test_df[label_col]  = test_df[label_col].astype(int) - 1

NUM_LABELS  = 5
NUM_CLASSES = 5

### Tokenizer & Datasets

In [ ]:
datasets = build_tf_datasets(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    tokenizer=tokenizer,
    text_col=text_col,
    label_col=label_col,
    max_length=MAX_LENGTH,
)

### Model, Class Weights & Trainer

The Video Game split is ~58% 5-star, so we use inverse-frequency class weights via `WeightedTrainer` and enable label smoothing through `TrainingArguments.label_smoothing_factor`.

In [ ]:
BATCH_SIZE      = 32
NUM_WORKERS     = min(4, os.cpu_count() or 0)
NUM_EPOCHS      = 5
LEARNING_RATE   = 2e-5
WARMUP_RATIO    = 0.1
LABEL_SMOOTHING = 0.1
RUN_NAME        = 'DistilBERT VG'
OUT_DIR         = MODELS_DIR / 'distilbert_vg'

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

# Inverse-frequency class weights to counter the ~58% 5-star imbalance
class_counts = np.bincount(np.asarray(train_df[label_col], dtype=np.int64))
class_weights = torch.tensor(
    (1.0 / class_counts) / (1.0 / class_counts).sum() * len(class_counts),
    dtype=torch.float,
)

wandb.init(
    name=RUN_NAME,
    tags=['Task 1.2', 'DistilBERT', 'Transformer', 'VG', 'Trainer'],
    config={
        'dataset':          'Video Games',
        'model_name':       MODEL_NAME,
        'optimizer':        'AdamW',
        'lr':               LEARNING_RATE,
        'epochs':           NUM_EPOCHS,
        'batch_size':       BATCH_SIZE,
        'max_length':       MAX_LENGTH,
        'warmup_ratio':     WARMUP_RATIO,
        'label_smoothing':  LABEL_SMOOTHING,
        'class_weights':    class_weights.tolist(),
    },
    reinit=True,
    mode='online' if LOG_WANDB else 'disabled',
)

training_args = TrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=1.0,
    label_smoothing_factor=LABEL_SMOOTHING,
    fp16=torch.cuda.is_available(),
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_strategy='epoch',
    dataloader_num_workers=NUM_WORKERS,
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to=['wandb'] if LOG_WANDB else 'none',
    run_name=RUN_NAME,
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets['train'],
    eval_dataset=datasets['val'],
    compute_metrics=compute_metrics_tf,
    processing_class=tokenizer,
    class_weights=class_weights,
)

trainer.train()
trainer.save_model(str(OUT_DIR))

wandb.finish()

### Test-Set Evaluation & Confusion Matrix

In [ ]:
_ = evaluate_tf(
    trainer,
    datasets['test'],
    label='DistilBERT-VG',
    class_names=['1-star', '2-star', '3-star', '4-star', '5-star'],
)

In [ ]:
plot_confusion_matrix_tf(
    trainer,
    datasets['test'],
    NUM_CLASSES,
    ['1-star', '2-star', '3-star', '4-star', '5-star'],
    'DistilBERT Confusion Matrix — Video Game Reviews',
    normalize=True,
)